install necessary libraries if they didn't

In [ ]:
# pip install earthengine-api geemap pycrs pathlib geopandas

# Adding 'field_id'

'field_id' is a field in your shapefile that helps to combine meteorological and spectral data as well as make an output map

In [ ]:
import geopandas as gpd

base_path=Path('path/to/your/shapefile')
gdf = gpd.read_file(file)
gdf['field_id'] = gdf.index
gdf.to_file(base_path.joinpath(f"path/to/your/shapefile"))

# Authentication in GEE

In [ ]:
import ee
import geemap
from pathlib import Path


ee.Authenticate() 
ee.Initialize(project='your_project_name', opt_url='https://earthengine-highvolume.googleapis.com')

## Downloading spectral data

In [ ]:
def mask_l8_clouds(image):
    """Masks clouds and cloud shadows in a Landsat 8 SR image using QA_PIXEL.

  Args:
    image (ee.Image): A Landsat 8 SR image (Collection 2, Level 2).

  Returns:
    ee.Image: A cloud-masked Landsat 8 image with scaled reflectance
              and original metadata (including system:time_start).
  """
    qa = image.select('QA_PIXEL')
    
    dilated_cloud_bit_mask = 1 << 1
    cloud_shadow_bit_mask = 1 << 4
    cloud_bit_mask = 1 << 3
    snow_mask= 1 << 5
    mask = (
      qa.bitwiseAnd(dilated_cloud_bit_mask).eq(0)
      .And(qa.bitwiseAnd(cloud_shadow_bit_mask).eq(0))
      .And(qa.bitwiseAnd(cloud_bit_mask).eq(0))
    )
    return image.updateMask(mask)\
      .select("SR_B.*").divide(10000)\
        .set('date', image.date().format('YYYY-MM-dd')).copyProperties(image, ["system:time_start", "system:index"]) 
    

def add_indices_landsat(image):
    red=image.select('SR_B4').rename('red')
    nir=image.select('SR_B5').rename('nir')
    blue=image.select('SR_B2').rename('blue')
    swir1=image.select('SR_B6').rename('swir1')
    green=image.select('SR_B3').rename('green')
    swir2=image.select('SR_B7').rename('swir2')
 
    return image.addBands([
        red, nir, blue, swir1, green, swir2
    ])

def add_indices_landsat5(image):
    red = image.select('SR_B3').rename('red')
    nir = image.select('SR_B4').rename('nir')
    blue = image.select('SR_B1').rename('blue')
    green = image.select('SR_B2').rename('green')
    swir1 = image.select('SR_B5').rename('swir1')
    swir2 = image.select('SR_B7').rename('swir2')
    return image.addBands([red, nir, blue, swir1, green, swir2])

year=2015
path_to_shp=Path(f'path/to/your/shapefile')
shape=geemap.shp_to_ee(path_to_shp)
start_date = f"{year}-04-01"
end_date = f"{year}-11-01"

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterDate(start_date, end_date)
      .filterBounds(shape)
      .filter(ee.Filter.lt('CLOUD_COVER', 50))
      .map(mask_l8_clouds)
      .map(add_indices_landsat))

l5 = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
      .filterDate(start_date, end_date)
      .filterBounds(shape)
      .filter(ee.Filter.lt('CLOUD_COVER', 50))
      .map(mask_l8_clouds)
      .map(add_indices_landsat5))

l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
  .filterDate(start_date, end_date)
  .filterBounds(shape)
  .filter(ee.Filter.lt('CLOUD_COVER', 50))
  .map(mask_l8_clouds)
  .map(add_indices_landsat))

collection = l5.merge(l8).merge(l9)
band_image = collection.select(['red', 'nir', 'blue', 'swir1', 'green', 'swir2'])
band_image = band_image.toBands()
geemap.zonal_statistics(band_image, shape, f'{year}.csv', statistics_type='MEAN', scale=30)

## Dowloading meteorological data

In [ ]:
def filter_bands(image):
    """Фильтрует изображения, добавляя температуру и осадки."""
    temperature = image.select('temperature_2m').rename('temperature')
    precipitation = image.select('total_precipitation_sum').rename('precipitation')

    return image.addBands([temperature, precipitation]).set('date', image.date().format('YYYY-MM-dd'))

year=2015
path_to_shp=Path(f'path/to/your/shapefile')
shape=geemap.shp_to_ee(path_to_shp)
start_date = f"{year}-04-01"
end_date = f"{year}-11-01"

dataset=(ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
                .filter(ee.Filter.date(start_date, end_date))
                .filterBounds(shape)
                .map(filter_bands))

meteo_image = collection.select(['temperature', 'precipitation'])
meteo_image = meteo_image.toBands()
geemap.zonal_statistics(meteo_image, shape, f'{year}.csv', statistics_type='MEAN', scale=30)